# RQ3: How are automated security tools adopted in software development, and what challenges do developers face in their use?

Part 1: frequency of use of the automated security tools, by tool category and period (daily, weekly, monthly, annually, not applicable). The challenges, desired improvements and future tools are analyzed in `RQ3_part2_analysis.ipynb`.

Outputs (`results/RQ3/`): one CSV per period and category (`daily/`, `weekly/`, `monthly/`, `annually/`, `not_applicable/`) and the summary `summary_period_category_tools.xlsx`.

## Pre-Processing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df_o = pd.read_excel('data/qualitative_responses_ptbr.xlsx')

df = df_o[:]

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('\n', '', regex=True)

print(len(df))

print(df.columns.tolist())

95
['ID', 'Carimbo de data/hora', 'Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro que sou maior de 18 anos e aceito participar por meio do formulário de participação, conforme as disposições da Lei Geral de Proteção de Dados (Lei nº 13.709/2018).Em caso de resposta negativa, o formulário será encerrado, garantindo assim o respeito à autonomia e privacidade dos participantes, conforme estabelecido pela legislação vigente.', 'Qual é a sua idade?', 'Em qual estado do Brasil você mora atualmente?', 'Qual é o seu gênero?', 'Qual é o seu nível de educação formal concluída?', 'Qual seu modelo de trabalho?', 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?', 'Por favor, indique a senioridade da posição que ocupa.', 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?', 'Quantos anos de experiência você tem em funções relacionadas a Segurança de 

In [3]:
df = df.rename(columns={
    'Quais das seguintes ferramentas específicas de segurança de software você utiliza atualmente ou já utilizou anteriormente?':
    'tools_used'
})

df_tools = df[['tools_used']]
df_tools.insert(0, "Respondent", range(1, len(df_tools) + 1))
df_tools.head()

,Respondent,tools_used
0,1,AppScan
1,2,NaN
2,3,"SonarQube, Checkmarx, Xray"
3,4,"Sonatype, Ansible, Puppet"
4,5,"Snyk, Fortify"


In [4]:
categories = {
    "3rd Party Vulnerability Scanning": [
        "Dependabot", "PHP Security Checker", "Snyk", "Clair", "Blackduck",
        "Retire.JS", "SRC:CLR", "Anchore", "Node Security Platform", "Trivy",
        "Sonatype", "Whitesource"
    ],
    "Static Application Security Testing": [  # SAST
        "CodeQL", "CheckStyle", "Coverity", "SonarQube", "Bandit", "PHPStan",
        "LGTM", "Brakeman", "ESLint Security", "Dart Analyzer", "DevSkim"
    ],
    "Dynamic Application Security Testing": [  # DAST
        "Qark", "Checkmarx", "CFN NAG"
    ],
    "Secure Configuration/Hardening": [
        "Chef", "Ansible", "Terraform", "Puppet", "Phan", "Seccomp",
        "CloudCustodian", "Turbot"
    ],
    "Compliance/Hardening Checks": [
        "Bane", "Ansible-Lint", "Blackbox", "Puppet-Lint", "InSpec",
        "Kube Audit", "Foodcritic", "Cookstyle", "conftest"
    ],
    "Secrets Management": [
        "Pinterest Knox", "GitLeaks", "CyberArk Conjur",
        "ThoughtWorks Talisman", "Berglas", "Docker Secrets", "Trufflehog"
    ]
}

In [5]:
tool_cols = [
    col for col in df.columns
    if "categorias de ferramentas" in str(col).lower() 
]

print(f"\n{len(tool_cols)} columns found for the tools question:")
for c in tool_cols:
    print("-", c)

df_categories = df[tool_cols].copy()

# iterate over the columns and rename if they contain one of the category keys
new_names = {}

for col in df_categories.columns:
    for key in categories.keys():
        if key.lower() in col.lower():  # compare ignoring case
            new_names[col] = key
            break  # avoid overwriting if more than one key appears

# apply the renaming
df_categories = df_categories.rename(columns=new_names)
df_categories.head()



6 columns found for the tools question:
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [3rd Party Vulnerability Scanning Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Static Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicá

,3rd Party Vulnerability Scanning,Static Application Security Testing,Dynamic Application Security Testing,Secure Configuration/Hardening,Compliance/Hardening Checks,Secrets Management
0,Não Aplicável,Mensalmente,Diariamente,Não Aplicável,Não Aplicável,Anualmente
1,NaN,NaN,NaN,NaN,NaN,NaN
2,Não Aplicável,Semanalmente,Não Aplicável,Semanalmente,Não Aplicável,Semanalmente
3,Diariamente,Diariamente,Não Aplicável,Não Aplicável,Não Aplicável,Diariamente
4,Diariamente,Não Aplicável,Diariamente,Não Aplicável,Não Aplicável,Diariamente


In [6]:
likert_scale = [
    "Diariamente",
    "Semanalmente",
    "Mensalmente",
    "Anualmente",
    "Não Aplicável",
]

In [7]:
df_categories.insert(0, "Respondent", range(1, len(df_categories) + 1))

In [8]:
df_categories.head()

,Respondent,3rd Party Vulnerability Scanning,Static Application Security Testing,Dynamic Application Security Testing,Secure Configuration/Hardening,Compliance/Hardening Checks,Secrets Management
0,1,Não Aplicável,Mensalmente,Diariamente,Não Aplicável,Não Aplicável,Anualmente
1,2,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Não Aplicável,Semanalmente,Não Aplicável,Semanalmente,Não Aplicável,Semanalmente
3,4,Diariamente,Diariamente,Não Aplicável,Não Aplicável,Não Aplicável,Diariamente
4,5,Diariamente,Não Aplicável,Diariamente,Não Aplicável,Não Aplicável,Diariamente


## Saving Daily Tool Frequency per Category

In [9]:
results = {}

for category, list_tools in categories.items():
    selected = df_categories[['Respondent',category]]
    df_merged = selected.merge(df_tools, on='Respondent', how='inner')
    print(f"\nProcessing category: {category}")


    tools_col = f"tools_{category}"

    # 1) Create a column filtering the tools by category.
    def tools_filter(text):
        if pd.isna(text):
            return None
        tools = [t.strip() for t in text.split(",")]
        filtereds = [f for f in tools if f in list_tools]
        return ", ".join(filtereds) if filtereds else None

    df_merged[tools_col] = df_merged["tools_used"].apply(tools_filter)

    # 2) Filter only those who use the "Diariamente" category.
    df_filtered = df_merged[
        (df_merged[category] == "Diariamente") &
        (df_merged[tools_col].notna())
    ].copy()

    # 3) Exploding tools
    df_filtered["list"] = df_filtered[tools_col].apply(lambda x: [t.strip() for t in x.split(",")])
    df_exploded = df_filtered.explode("list")
    frequencies = df_exploded["list"].value_counts()

    # 4) Automatic name
    name_var = f"frequencies_daily_{category.replace(' ', '_').replace('/', '_').lower()}"

    # Store in the final dictionary
    results[name_var] = frequencies

    print(f"-> Frequencies saved: {name_var}")



Processing category: 3rd Party Vulnerability Scanning
-> Frequencies saved: frequencies_daily_3rd_party_vulnerability_scanning

Processing category: Static Application Security Testing
-> Frequencies saved: frequencies_daily_static_application_security_testing

Processing category: Dynamic Application Security Testing
-> Frequencies saved: frequencies_daily_dynamic_application_security_testing

Processing category: Secure Configuration/Hardening
-> Frequencies saved: frequencies_daily_secure_configuration_hardening

Processing category: Compliance/Hardening Checks
-> Frequencies saved: frequencies_daily_compliance_hardening_checks

Processing category: Secrets Management
-> Frequencies saved: frequencies_daily_secrets_management


In [10]:
import os
import pandas as pd

os.makedirs("results/RQ3/daily", exist_ok=True)

for name, serie in results.items():
    output_df = serie.reset_index()
    output_df.columns = ["tool", "frequency"]
    
    path = f"results/RQ3/daily/{name}.csv"
    
    output_df.to_csv(path, index=False)
    print(f"✔ CSV saved: {path}")



✔ CSV saved: results/RQ3/daily/frequencies_daily_3rd_party_vulnerability_scanning.csv
✔ CSV saved: results/RQ3/daily/frequencies_daily_static_application_security_testing.csv
✔ CSV saved: results/RQ3/daily/frequencies_daily_dynamic_application_security_testing.csv
✔ CSV saved: results/RQ3/daily/frequencies_daily_secure_configuration_hardening.csv
✔ CSV saved: results/RQ3/daily/frequencies_daily_compliance_hardening_checks.csv
✔ CSV saved: results/RQ3/daily/frequencies_daily_secrets_management.csv


## Saving Weekly Tool Frequency per Category

In [11]:
results = {}

for category, list_tools in categories.items():
    df_selected = df_categories[['Respondent',category]]
    df_merged = df_selected.merge(df_tools, on='Respondent', how='inner')
    print(f"\nProcessing category: {category}")

    def filter_tools(text):
        if pd.isna(text):
            return None
        tools = [t.strip() for t in text.split(",")]
        filtereds = [f for f in tools if f in list_tools]
        return ", ".join(filtereds) if filtereds else None

    df_merged[tools_col] = df_merged["tools_used"].apply(filter_tools)

    df_filtered = df_merged[
        (df_merged[category] == "Semanalmente") &
        (df_merged[tools_col].notna())
    ].copy()

    df_filtered["list"] = df_filtered[tools_col].apply(lambda x: [t.strip() for t in x.split(",")])
    df_exploded = df_filtered.explode("list")
    frequencies = df_exploded["list"].value_counts()

    var_name = f"frequencies_weekly_{category.replace(' ', '_').replace('/', '_').lower()}"

    results[var_name] = frequencies

    print(f"-> Frequencies Saved: {var_name}")


Processing category: 3rd Party Vulnerability Scanning
-> Frequencies Saved: frequencies_weekly_3rd_party_vulnerability_scanning

Processing category: Static Application Security Testing
-> Frequencies Saved: frequencies_weekly_static_application_security_testing

Processing category: Dynamic Application Security Testing
-> Frequencies Saved: frequencies_weekly_dynamic_application_security_testing

Processing category: Secure Configuration/Hardening
-> Frequencies Saved: frequencies_weekly_secure_configuration_hardening

Processing category: Compliance/Hardening Checks
-> Frequencies Saved: frequencies_weekly_compliance_hardening_checks

Processing category: Secrets Management
-> Frequencies Saved: frequencies_weekly_secrets_management


In [12]:
os.makedirs("results/RQ3/weekly", exist_ok=True)

for name, serie in results.items():
    output_df = serie.reset_index()
    output_df.columns = ["tool", "frequency"]
    
    path = f"results/RQ3/weekly/{name}.csv"
    
    output_df.to_csv(path, index=False)
    print(f"✔ CSV saved: {path}")



✔ CSV saved: results/RQ3/weekly/frequencies_weekly_3rd_party_vulnerability_scanning.csv
✔ CSV saved: results/RQ3/weekly/frequencies_weekly_static_application_security_testing.csv
✔ CSV saved: results/RQ3/weekly/frequencies_weekly_dynamic_application_security_testing.csv
✔ CSV saved: results/RQ3/weekly/frequencies_weekly_secure_configuration_hardening.csv
✔ CSV saved: results/RQ3/weekly/frequencies_weekly_compliance_hardening_checks.csv
✔ CSV saved: results/RQ3/weekly/frequencies_weekly_secrets_management.csv


## Saving Monthly Tool Frequency per Category

In [13]:
results = {}

for category, tool_list in categories.items():
    df_selected = df_categories[['Respondent', category]]
    df_merged = df_selected.merge(df_tools, on='Respondent', how='inner')
    print(f"\nProcessing category: {category}")

    tools_col = f"tools_{category}"

    # 1) Create column by filtering tools from the category
    def filter_tools(text):
        if pd.isna(text):
            return None
        tools = [t.strip() for t in text.split(",")]
        filtered = [t for t in tools if t in tool_list]
        return ", ".join(filtered) if filtered else None

    df_merged[tools_col] = df_merged["tools_used"].apply(filter_tools)

    # 2) Filter only respondents who use the category "Mensalmente"
    df_filtered = df_merged[
        (df_merged[category] == "Mensalmente") &
        (df_merged[tools_col].notna())
    ].copy()

    # 3) Explode tools and count frequencies
    df_filtered["list"] = df_filtered[tools_col].apply(
        lambda x: [t.strip() for t in x.split(",")]
    )
    df_exploded = df_filtered.explode("list")
    frequencies = df_exploded["list"].value_counts()

    # 4) Automatic name for the result
    var_name = (
        f"frequencies_monthly_{category.replace(' ', '_').replace('/', '_').lower()}"
    )

    # Store in the final dictionary
    results[var_name] = frequencies

    print(f"-> Frequencies saved as: {var_name}")



Processing category: 3rd Party Vulnerability Scanning
-> Frequencies saved as: frequencies_monthly_3rd_party_vulnerability_scanning

Processing category: Static Application Security Testing
-> Frequencies saved as: frequencies_monthly_static_application_security_testing

Processing category: Dynamic Application Security Testing
-> Frequencies saved as: frequencies_monthly_dynamic_application_security_testing

Processing category: Secure Configuration/Hardening
-> Frequencies saved as: frequencies_monthly_secure_configuration_hardening

Processing category: Compliance/Hardening Checks
-> Frequencies saved as: frequencies_monthly_compliance_hardening_checks

Processing category: Secrets Management
-> Frequencies saved as: frequencies_monthly_secrets_management


In [14]:
os.makedirs("results/RQ3/monthly", exist_ok=True)

for name, serie in results.items():
    output_df = serie.reset_index()
    output_df.columns = ["tool", "frequency"]
    
    path = f"results/RQ3/monthly/{name}.csv"
    
    output_df.to_csv(path, index=False)
    print(f"✔ CSV saved: {path}")



✔ CSV saved: results/RQ3/monthly/frequencies_monthly_3rd_party_vulnerability_scanning.csv
✔ CSV saved: results/RQ3/monthly/frequencies_monthly_static_application_security_testing.csv
✔ CSV saved: results/RQ3/monthly/frequencies_monthly_dynamic_application_security_testing.csv
✔ CSV saved: results/RQ3/monthly/frequencies_monthly_secure_configuration_hardening.csv
✔ CSV saved: results/RQ3/monthly/frequencies_monthly_compliance_hardening_checks.csv
✔ CSV saved: results/RQ3/monthly/frequencies_monthly_secrets_management.csv


## Saving Annual Tool Frequency per Category

In [15]:
results = {}

for category, tool_list in categories.items():
    df_selected = df_categories[['Respondent', category]]
    df_merged = df_selected.merge(df_tools, on='Respondent', how='inner')
    print(f"\nProcessing category: {category}")

    tools_col = f"tools_{category}"

    # 1) Create column by filtering tools from the category
    def filter_tools(text):
        if pd.isna(text):
            return None
        tools = [t.strip() for t in text.split(",")]
        filtered = [t for t in tools if t in tool_list]
        return ", ".join(filtered) if filtered else None

    df_merged[tools_col] = df_merged["tools_used"].apply(filter_tools)

    # 2) Filter only respondents who use the category "Anualmente"
    df_filtered = df_merged[
        (df_merged[category] == "Anualmente") &
        (df_merged[tools_col].notna())
    ].copy()

    # 3) Explode tools and count frequencies
    df_filtered["list"] = df_filtered[tools_col].apply(
        lambda x: [t.strip() for t in x.split(",")]
    )
    df_exploded = df_filtered.explode("list")
    frequencies = df_exploded["list"].value_counts()

    # 4) Automatic name for the result
    var_name = (
        f"frequencies_annually_{category.replace(' ', '_').replace('/', '_').lower()}"
    )

    # Store in the final dictionary
    results[var_name] = frequencies

    print(f"-> Frequencies saved as: {var_name}")



Processing category: 3rd Party Vulnerability Scanning
-> Frequencies saved as: frequencies_annually_3rd_party_vulnerability_scanning

Processing category: Static Application Security Testing
-> Frequencies saved as: frequencies_annually_static_application_security_testing

Processing category: Dynamic Application Security Testing
-> Frequencies saved as: frequencies_annually_dynamic_application_security_testing

Processing category: Secure Configuration/Hardening
-> Frequencies saved as: frequencies_annually_secure_configuration_hardening

Processing category: Compliance/Hardening Checks
-> Frequencies saved as: frequencies_annually_compliance_hardening_checks

Processing category: Secrets Management
-> Frequencies saved as: frequencies_annually_secrets_management


In [16]:
os.makedirs("results/RQ3/annually", exist_ok=True)

for name, serie in results.items():
    output_df = serie.reset_index()
    output_df.columns = ["tool", "frequency"]

    path = f"results/RQ3/annually/{name}.csv"
    
    output_df.to_csv(path, index=False)
    print(f"✔ CSV saved: {path}")



✔ CSV saved: results/RQ3/annually/frequencies_annually_3rd_party_vulnerability_scanning.csv
✔ CSV saved: results/RQ3/annually/frequencies_annually_static_application_security_testing.csv
✔ CSV saved: results/RQ3/annually/frequencies_annually_dynamic_application_security_testing.csv
✔ CSV saved: results/RQ3/annually/frequencies_annually_secure_configuration_hardening.csv
✔ CSV saved: results/RQ3/annually/frequencies_annually_compliance_hardening_checks.csv
✔ CSV saved: results/RQ3/annually/frequencies_annually_secrets_management.csv


## Saving Not Applicable Tool Frequency per Category


In [17]:
results = {}

for category, tool_list in categories.items():
    df_selected = df_categories[['Respondent', category]]
    df_merged = df_selected.merge(df_tools, on='Respondent', how='inner')
    print(f"\nProcessing category: {category}")

    tools_col = f"tools_{category}"

    # 1) Create column by filtering tools from the category
    def filter_tools(text):
        if pd.isna(text):
            return None
        tools = [t.strip() for t in text.split(",")]
        filtered = [t for t in tools if t in tool_list]
        return ", ".join(filtered) if filtered else None

    df_merged[tools_col] = df_merged["tools_used"].apply(filter_tools)

    # 2) Filter only respondents who use the category "Não Aplicável" (Not Applicable)
    df_filtered = df_merged[
        (df_merged[category] == "Não Aplicável") &
        (df_merged[tools_col].notna())
    ].copy()

    # 3) Explode tools and count frequencies
    df_filtered["list"] = df_filtered[tools_col].apply(
        lambda x: [t.strip() for t in x.split(",")]
    )
    df_exploded = df_filtered.explode("list")
    frequencies = df_exploded["list"].value_counts()

    # 4) Automatic name for the result
    var_name = (
        f"frequencies_not_applicable_{category.replace(' ', '_').replace('/', '_').lower()}"
    )

    # Store in the final dictionary
    results[var_name] = frequencies

    print(f"-> Frequencies saved as: {var_name}")



Processing category: 3rd Party Vulnerability Scanning
-> Frequencies saved as: frequencies_not_applicable_3rd_party_vulnerability_scanning

Processing category: Static Application Security Testing
-> Frequencies saved as: frequencies_not_applicable_static_application_security_testing

Processing category: Dynamic Application Security Testing
-> Frequencies saved as: frequencies_not_applicable_dynamic_application_security_testing

Processing category: Secure Configuration/Hardening
-> Frequencies saved as: frequencies_not_applicable_secure_configuration_hardening

Processing category: Compliance/Hardening Checks
-> Frequencies saved as: frequencies_not_applicable_compliance_hardening_checks

Processing category: Secrets Management
-> Frequencies saved as: frequencies_not_applicable_secrets_management


In [18]:
os.makedirs("results/RQ3/not_applicable", exist_ok=True)

for name, serie in results.items():
    output_df = serie.reset_index()
    output_df.columns = ["tool", "frequency"]
    
    path = f"results/RQ3/not_applicable/{name}.csv"
    
    output_df.to_csv(path, index=False)
    print(f"✔ CSV saved: {path}")



✔ CSV saved: results/RQ3/not_applicable/frequencies_not_applicable_3rd_party_vulnerability_scanning.csv
✔ CSV saved: results/RQ3/not_applicable/frequencies_not_applicable_static_application_security_testing.csv
✔ CSV saved: results/RQ3/not_applicable/frequencies_not_applicable_dynamic_application_security_testing.csv
✔ CSV saved: results/RQ3/not_applicable/frequencies_not_applicable_secure_configuration_hardening.csv
✔ CSV saved: results/RQ3/not_applicable/frequencies_not_applicable_compliance_hardening_checks.csv
✔ CSV saved: results/RQ3/not_applicable/frequencies_not_applicable_secrets_management.csv


## Saving summary spreadsheet

In [19]:
folders = {
    "daily": "daily",
    "weekly": "weekly",
    "monthly": "monthly",
    "annually": "annually",
    "not applicable": "not_applicable"
}

excel_categories = list(categories.keys())
excel_cols = list(folders.keys())

df_excel = pd.DataFrame(index=excel_categories, columns=excel_cols)

def csv_to_string(path):
    if not os.path.exists(path):
        print(path)
        return "df not exists"
    df = pd.read_csv(path)
    if df.empty:
        return ""
    
    text = "; ".join(f"{row['tool']}: {row['frequency']}" 
                      for _, row in df.iterrows())
    
    return text.replace("; ", ";\n")

for name_col, folder_name in folders.items():
    folder = os.path.join("results/RQ3", folder_name)  
    for category in excel_categories:
        name_csv = f"frequencies_{folder_name}_{category.replace(' ', '_').replace('/', '_').lower()}.csv"
        path = os.path.join(folder, name_csv)
        content = csv_to_string(path)

        df_excel.loc[category, name_col] = content

path = "results/RQ3/"
df_excel.to_excel(f"{path}summary_period_category_tools.xlsx", index=True)
print(df_excel)
print(f"✔ Spreadsheet saved: summary_period_category_tools.xlsx")

                                                                                  daily  \
3rd Party Vulnerability Scanning      Snyk: 18;\nTrivy: 15;\nDependabot: 15;\nSonaty...   
Static Application Security Testing   SonarQube: 29;\nCodeQL: 12;\nESLint Security: ...   
Dynamic Application Security Testing                             Checkmarx: 5;\nQark: 1   
Secure Configuration/Hardening        Ansible: 9;\nTerraform: 8;\nPuppet: 4;\nSeccom...   
Compliance/Hardening Checks                     Kube Audit: 2;\nBlackbox: 1;\nInSpec: 1   
Secrets Management                    GitLeaks: 16;\nDocker Secrets: 10;\nTrufflehog...   

                                                                                 weekly  \
3rd Party Vulnerability Scanning      Snyk: 7;\nDependabot: 6;\nTrivy: 4;\nRetire.JS...   
Static Application Security Testing   SonarQube: 11;\nCheckStyle: 1;\nESLint Securit...   
Dynamic Application Security Testing                             Checkmarx: 5;\nQark: 1  